# Run Gemma 4 E4B model with `vLLM`

As you have seen in the last notebooks, the speed of the models depends on the number of 
(active) parameters. In model inference, the speed is dominated by
the availabe memory bandwidth. This is the main reason why GPUs are so much faster in 
text generation compared to CPUs (in addition to prompt parsing).

However, if we can reduce the size of the parameters (not just the number), we could also get
speed increases. This can be achieved by quantization. Unfortunately, `transformers` do not
support quantized models. Therefore, we have to use another software which is optimized for
that.

`vllm` can be used both as a library and as an Open AI compatible REST server. In this
notebook, we take the endpoint approach, so many notebooks can access the same server.

First, run `vllm serve cyankiwi/gemma-4-E4B-it-AWQ-INT4 --port 8000 --reasoning-parser gemma4 --language-model-only --chat-template tool_chat_template_gemma4.jinja`

In [ ]:
!nvidia-smi

vLLM offers an Open AI compatible API:

In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="secret")

We want to use the notebook for different models:

In [ ]:
model = "cyankiwi/gemma-4-E4B-it-AWQ-INT4"

## Chat Completion API

Use the chat completion API first:

In [ ]:
completion = client.chat.completions.create(
    model=model, 
    messages=[{ "role": "user",
                "content": "How many 'r's are in 'strawberry'?" } ]
)

print(completion.choices[0].message.content)

The message does not consist of `content` only:

In [ ]:
completion

In [ ]:
from IPython.display import display, Markdown
display(Markdown(completion.choices[0].message.reasoning))

In [ ]:
display(Markdown(completion.choices[0].message.content))

## Responses API

Now switch to the more modern responses API

In [ ]:
response = client.responses.create(model=model, 
                                   input="How many 'r's are in 'strawberry'?")
display(Markdown(response.output_text))

Examine the response:

In [ ]:
response

In [ ]:
display(Markdown(response.output[0].content[0].text))

In [ ]:
display(Markdown(response.output[1].content[0].text))

Now try to disable thinking by giving the appropriate argument:

In [ ]:
completion = client.chat.completions.create(
    model=model, 
    messages=[{ "role": "user",
                "content": "How many 'r's are in 'strawberry'?" } ],
    extra_body={ "chat_template_kwargs": {"enable_thinking": None} }
)

completion

Unfortunately, this does not work.

In [ ]:
completion = client.chat.completions.create(
    model=model, 
    messages=[{ "role": "user",
                "content": "How many 'r's are in 'strawberry'?" } ],
    extra_body={ "reasoning_effort": "none", "include_reasoning": False }
)

completion

This does not show reasoning, but it is anyway included (see number of tokens).